# 02 — Feature Engineering (deterministic)

Derives new numeric features from the cleaned columns — the raw material every downstream model needs.
No ML yet: ratios, spreads, diversity/entropy, completeness, tiers, and interaction terms.
Reads `artifacts/localities_clean.parquet`, writes `artifacts/features_base.parquet`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ART = Path.cwd() / "artifacts"
df = pd.read_parquet(ART / "localities_clean.parquet")
print("Loaded:", df.shape)

NUM_COLS = [c for c in df.columns if c.startswith("num_")]
print("Amenity count columns:", NUM_COLS)

Loaded: (1001, 46)
Amenity count columns: ['num_educational_institute', 'num_hospital', 'num_shopping_centre', 'num_transportation_hub', 'num_commercial_hub', 'num_tourist_spot', 'num_nearby_localities']


In [2]:
# 1) Price-derived features
# buy/rent ratio ~ a rental-yield proxy (higher = pricier to buy vs rent)
df["buy_rent_ratio"] = df["res_avg_buy"] / df["res_avg_rent"]
# price spread = uncertainty / heterogeneity within the locality
df["price_spread"] = df["res_max_buy"] - df["res_min_buy"]
df["price_spread_pct"] = df["price_spread"] / df["res_avg_buy"]
# affluence tier from quantiles of avg buy price (labelled, NaN-safe)
df["affluence_tier"] = pd.qcut(df["res_avg_buy"], q=4,
                               labels=["Value", "Mid", "Premium", "Ultra-Premium"])
print(df[["res_avg_buy", "buy_rent_ratio", "price_spread", "affluence_tier"]].describe(include="all").to_string())

         res_avg_buy  buy_rent_ratio  price_spread affluence_tier
count     549.000000      549.000000    549.000000            549
unique           NaN             NaN           NaN              4
top              NaN             NaN           NaN          Value
freq             NaN             NaN           NaN            141
mean    14625.318761      399.854787   6837.522769            NaN
std     11087.004794      118.652730   5293.100286            NaN
min      3100.000000      182.352941   1500.000000            NaN
25%      7350.000000      317.910448   3400.000000            NaN
50%     10850.000000      368.000000   5300.000000            NaN
75%     17750.000000      469.444444   8300.000000            NaN
max     77800.000000     1297.619048  38600.000000            NaN


In [3]:
# 2) Amenity-derived features
# total amenities present (sum of de-duplicated counts)
df["total_amenities"] = df[NUM_COLS].sum(axis=1)

# amenity diversity = Shannon entropy across amenity types (0 = one type only, high = balanced mix)
def shannon_entropy(row):
    counts = np.array([row[c] for c in NUM_COLS], dtype=float)
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts[counts > 0] / total
    return float(-(p * np.log2(p)).sum())

df["amenity_diversity"] = df[NUM_COLS].apply(shannon_entropy, axis=1)

# infra completeness = share of the 7 amenity TYPES that are present (>=1)
df["infra_completeness"] = (df[NUM_COLS] > 0).sum(axis=1) / len(NUM_COLS)

# retail-vs-residential balance: shopping+commercial vs education+hospital (civic)
retail = df["num_shopping_centre"] + df["num_commercial_hub"]
civic = df["num_educational_institute"] + df["num_hospital"]
df["retail_civic_balance"] = (retail - civic) / (retail + civic).replace(0, np.nan)

print(df[["total_amenities", "amenity_diversity", "infra_completeness", "retail_civic_balance"]].describe().round(2).to_string())

       total_amenities  amenity_diversity  infra_completeness  retail_civic_balance
count          1001.00            1001.00             1001.00                782.00
mean             11.12               1.22                0.44                 -0.35
std               7.32               0.79                0.28                  0.59
min               0.00               0.00                0.00                 -1.00
25%               8.00               0.54                0.29                 -1.00
50%              11.00               1.28                0.43                 -0.43
75%              15.00               1.88                0.71                  0.00
max              38.00               2.73                1.00                  1.00


In [4]:
# 3) Interaction features (give downstream models non-linear combinations up front)
# Use median-imputed, scaled copies ONLY for interactions so NaNs don't propagate everywhere.
for base in ["res_avg_buy", "total_amenities", "infra_completeness", "num_transportation_hub"]:
    z = (df[base] - df[base].mean()) / df[base].std(ddof=0)
    df[base + "_z"] = z

df["affluence_x_amenities"] = df["res_avg_buy_z"] * df["total_amenities_z"]
df["transport_x_infra"] = df["num_transportation_hub_z"] * df["infra_completeness_z"]
print("Interaction features added:",
      [c for c in df.columns if c.endswith("_z") or "_x_" in c])

Interaction features added: ['res_avg_buy_z', 'total_amenities_z', 'infra_completeness_z', 'num_transportation_hub_z', 'affluence_x_amenities', 'transport_x_infra']


In [5]:
# 3b) Employer quality (Improvement 2). Word-boundary match so short tokens
# (ey, sap, hcl, aon) don't match inside other words like "valley"/"journey".
import re
EMPLOYER_TIERS = {
    5: ["mckinsey", "bcg", "bain", "deloitte", "kpmg", "pwc", "ernst & young", "ernst and young", "e&y", "ey"],
    4: ["google", "american express", "oracle", "dell", "samsung", "microsoft", "amazon", "adobe", "sap"],
    3: ["tcs", "infosys", "wipro", "cognizant", "capgemini", "accenture", "tech mahindra", "hcl", "mindtree", "mphasis"],
    2: ["genpact", "concentrix", "aon", "xceedance", "ntt data", "xerox", "fidelity", "boston scientific"],
    1: ["hero motocorp", "maruti suzuki", "honda", "suzuki motorcycle", "carrier air conditioning"],
}

def _has(name, t):
    return re.search(r"(?<![a-z])" + re.escape(name) + r"(?![a-z])", t) is not None

def employer_quality_score(text):
    if pd.isna(text):
        return 0.0
    t = str(text).lower()
    return float(sum(w for w, names in EMPLOYER_TIERS.items() for name in names if _has(name, t)))

def employer_tier_max(text):
    if pd.isna(text):
        return 0
    t = str(text).lower()
    for tier in [5, 4, 3, 2, 1]:
        if any(_has(name, t) for name in EMPLOYER_TIERS[tier]):
            return tier
    return 0

df["employer_quality"] = df["Nearby employment hubs"].apply(employer_quality_score)
df["employer_tier_max"] = df["Nearby employment hubs"].apply(employer_tier_max)

# 3c) Rental yield (Improvement 7) - DESCRIPTIVE / segmentation only.
# NOTE: contains the buy price in its denominator, so it is NOT used as an NB06 imputation
# predictor (that would be target leakage + it is undefined exactly where buy is missing).
df["rental_yield_pct"] = (df["res_avg_rent"] * 12) / df["res_avg_buy"] * 100
df["renter_heavy"] = df["rental_yield_pct"] > 3.5

# 3d) Competitive density / choice intensity (Improvement 11)
amenity_present = (df[NUM_COLS] > 0).sum(axis=1)
df["choice_intensity"] = df["total_amenities"] / amenity_present.replace(0, np.nan)

def gini(row):
    counts = np.array([row[c] for c in NUM_COLS], dtype=float)
    counts = counts[counts > 0]
    if len(counts) == 0:
        return np.nan
    n = len(counts); mean_c = counts.mean()
    if mean_c == 0:
        return 0.0
    return float(np.sum(np.abs(counts[:, None] - counts[None, :])) / (2 * n * n * mean_c))

df["amenity_gini"] = df[NUM_COLS].apply(gini, axis=1)

print("employer_quality>0:", int((df.employer_quality > 0).sum()),
      "| tier_max:", df.employer_tier_max.value_counts().sort_index().to_dict())
print("corr(employer_quality, res_avg_buy):",
      round(df[["employer_quality", "res_avg_buy"]].corr().iloc[0, 1], 3))

employer_quality>0: 296 | tier_max: {0: 705, 1: 15, 2: 4, 3: 180, 4: 51, 5: 46}
corr(employer_quality, res_avg_buy): -0.204


In [6]:
# 4) Quality / sanity checks before saving
n = len(df)
new_feats = ["buy_rent_ratio", "price_spread", "price_spread_pct", "affluence_tier",
             "total_amenities", "amenity_diversity", "infra_completeness",
             "retail_civic_balance", "affluence_x_amenities", "transport_x_infra"]
report = pd.DataFrame({
    "non_null": [df[c].notna().sum() for c in new_feats],
    "pct_present": [round(df[c].notna().mean() * 100, 1) for c in new_feats],
}, index=new_feats)
print("New feature coverage:")
print(report.to_string())

# correlation of the numeric new features with affluence (where present)
corr = df[["res_avg_buy", "total_amenities", "amenity_diversity",
           "infra_completeness", "num_transportation_hub"]].corr()["res_avg_buy"].round(2)
print("\nCorrelation with res_avg_buy:")
print(corr.to_string())

New feature coverage:
                       non_null  pct_present
buy_rent_ratio              549         54.8
price_spread                549         54.8
price_spread_pct            549         54.8
affluence_tier              549         54.8
total_amenities            1001        100.0
amenity_diversity          1001        100.0
infra_completeness         1001        100.0
retail_civic_balance        782         78.1
affluence_x_amenities       549         54.8
transport_x_infra          1001        100.0

Correlation with res_avg_buy:
res_avg_buy               1.00
total_amenities           0.05
amenity_diversity         0.04
infra_completeness        0.03
num_transportation_hub    0.12


In [7]:
# 5) Persist
out = ART / "features_base.parquet"
df.to_parquet(out, index=False)
print("Saved", df.shape[0], "rows x", df.shape[1], "cols ->", out.relative_to(Path.cwd()))

Saved 1001 rows x 66 cols -> artifacts\features_base.parquet


## Output
`artifacts/features_base.parquet` — adds price ratios/spread/tier, amenity diversity (entropy),
infra completeness, retail-vs-civic balance, and interaction terms.

**Next:** `03_text_mining.ipynb` — NER, keyword/sector tagging, sentence embeddings, and topic modelling
on the five free-text columns (Path A: sentence-transformers + zero-shot + BERTopic).